# HateGuard - Deploy on Google Colab

This notebook allows anyone to run the full HateGuard system on their device via Google Colab. 
It sets up the backend (FastAPI), the frontend (React/Vite), and opens a public tunnel to interact with the UI.

In [ ]:
# 1. Clone the repository
!git clone https://github.com/ATANU28-bit/hate-comment-dectection.git
%cd hate-comment-dectection

In [ ]:
# 2. Install dependencies & tools
!sudo apt update && sudo apt install ffmpeg -y
!pip install -r requirements.txt
!npm install -g localtunnel

%cd ui
!npm install
%cd ..

In [ ]:
import subprocess
import time
import sys
import os

print("Starting Backend...")
# Start Backend
backend = subprocess.Popen(["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Start localtunnel for Backend
lt_backend = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE)
backend_url = lt_backend.stdout.readline().decode('utf-8').strip().replace("your url is: ", "")

print(f"⭐ BACKEND URL: {backend_url}")

# Set Backend URL for Frontend
with open("ui/.env", "w", encoding="utf-8") as f:
    f.write(f"VITE_API_URL={backend_url}\n")

print("Starting Frontend...")
# Start Frontend
frontend = subprocess.Popen(["npm", "run", "dev", "--prefix", "ui", "--", "--host", "0.0.0.0", "--port", "5173"])
time.sleep(3)

# Start localtunnel for Frontend
lt_frontend = subprocess.Popen(["lt", "--port", "5173"], stdout=subprocess.PIPE)
frontend_url = lt_frontend.stdout.readline().decode('utf-8').strip().replace("your url is: ", "")

print("\n=========================================================")
print("🚀 YOUR APPLICATION IS READY!")
print(f"👉 OPEN THIS LINK FOR THE UI: {frontend_url}")
print("=========================================================\n")
print("⚠️ IMPORTANT: When you click the link, localtunnel will ask for an 'Endpoint IP'.")
print("Copy the IP address below and paste it into the website to grant access:")
os.system("curl -s ipv4.icanhazip.com")

try:
    frontend.wait() # Keep cells running
except KeyboardInterrupt:
    backend.terminate()
    frontend.terminate()
    lt_backend.terminate()
    lt_frontend.terminate()